# Build a Regression Evaluation Helper

Same idea as the classification eval, different metrics. Once you have a trained regressor, the question is: is it actually good, and is anything suspicious going on? RMSE and R² don't answer that on their own — an RMSE of 150 could be excellent or terrible depending on the target scale, and an R² of 0.99 could mean a great model or a leakage bug.

In this notebook we make one call to the LLM to flag anomalies in the metrics as this is one place where the numbers alone don't tell you what to do.

## 1 - Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -q google-genai pandas scikit-learn matplotlib python-dotenv


In [ ]:
import inspect
import json
import os
import re
import sys
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

sys.path.append(str(Path("../02-Preprocessing-Helpers").resolve()))
sys.path.append(str(Path("../03-Modeling-Helpers").resolve()))
from preprocessing_pipeline import preprocessing_pipeline
from regression_helper import regression_helper

warnings.filterwarnings("ignore")
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

GEN_CONFIG = {"temperature": 0.0, "seed": 42}
pd.set_option("display.max_colwidth", None)

In [ ]:
result = preprocessing_pipeline(
    raw_path="../../data/hr_analytics.csv",
    target_col="MonthlyIncome",
    task="regression",
    column_types={"Attrition": "binary"},
)
X_train, X_test = result.X_train_enc, result.X_test_enc
y_train = pd.to_numeric(pd.Series(result.y_train), errors="coerce")
y_test = pd.to_numeric(pd.Series(result.y_test), errors="coerce")

results = regression_helper(X_train, X_test, y_train, y_test)

## 2 - Compute Metrics

We compute RMSE, MAE, R², and a few target statistics (mean, std, min, max). The target stats matter — they give the LLM a ruler. An RMSE of 147 only means something when you know the target spans 3,400 to 17,500.

In [ ]:
y_pred = results["y_pred"]
residuals = y_test.reset_index(drop=True) - pd.Series(y_pred).reset_index(drop=True)

metrics = {
    "model": results["model_name"],
    "rmse": float(root_mean_squared_error(y_test, y_pred)),
    "mae": float(mean_absolute_error(y_test, y_pred)),
    "r2": float(r2_score(y_test, y_pred)),
    "target_mean": float(y_test.mean()),
    "target_std": float(y_test.std()),
    "target_min": float(y_test.min()),
    "target_max": float(y_test.max()),
}

pd.DataFrame([metrics])

## 3 - LLM Flags Anomalies

The metrics go to the LLM. Common regression red flags: RMSE much larger than target std, R² close to 0 or suspiciously close to 1, large MAE/RMSE gap suggesting outlier sensitivity. The LLM returns a list of flags with severity and an explanation that cites the actual numbers.

In [ ]:
FLAG_PROMPT = (
    "You are a senior data scientist auditing a regression model. "
    "Given a metrics dict, identify any red flags or anomalies. "
    "Common issues: RMSE much larger than target std, R2 close to 0 or negative, "
    "suspiciously high R2 suggesting overfitting, large gap between MAE and RMSE indicating outlier sensitivity. "
    "Return ONLY valid JSON: a list of objects with keys "
    "flag (string), severity (high/medium/low), explanation (string). "
    "No markdown fences. No text outside the JSON."
)


def flag_anomalies(metrics):
    content = f"Metrics: {json.dumps(metrics, indent=2)}"
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=content,
        config={"system_instruction": FLAG_PROMPT, **GEN_CONFIG},
    )
    text = (resp.text or "").strip()
    text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "").strip()
    return json.loads(text)


In [ ]:
flags = flag_anomalies(metrics)
pd.set_option("display.max_colwidth", None)
pd.DataFrame(flags)


## 4 - Save for Reuse

Package the helpers into a Python file so later lessons can import them.

In [ ]:
components = [
    "import inspect",
    "import json",
    "import os",
    "import re",
    "from pathlib import Path",
    "",
    "import pandas as pd",
    "from dotenv import load_dotenv",
    "from google import genai",
    "from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error",
    "",
    "PROJECT_ROOT = Path.cwd().resolve()",
    "load_dotenv(PROJECT_ROOT / '.env')",
    "api_key = os.getenv('GEMINI_API_KEY')",
    "client = genai.Client(api_key=api_key) if api_key else None",
    f"GEN_CONFIG = {repr(GEN_CONFIG)}",
    f"FLAG_PROMPT = {repr(FLAG_PROMPT)}",
    "",
    inspect.getsource(flag_anomalies),
]

with open("regression_eval_helper.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved regression_eval_helper.py")